# Phase 1: Offline Distillation Data Generation (Incremental Caching Version)
Notebook này hỗ trợ lưu tạm (checkpoint) từng câu trả lời để không bị mất dữ liệu khi API lỗi giữa chừng.

In [ ]:
# [QUAN TRỌNG] Kết nối với Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/Data_Phase1', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/Data_Phase1/rollouts_cache', exist_ok=True)
    print("Đã kết nối Google Drive và khởi tạo thư mục Cache!")
except:
    print("Không chạy trên môi trường Colab.")

In [ ]:
!pip install -q datasets jsonlines tqdm openai

## Cấu hình Token Xoay Vòng

In [ ]:
import os
import phase1_distillation.config as config
import importlib

# NHẬP CÁC TOKEN CỦA BẠN VÀO ĐÂY
os.environ['HF_TOKENS'] = 'token1,token2,token3'

importlib.reload(config)
print(f"Số lượng Token sẵn sàng xoay vòng: {len(config.HF_TOKENS)}")

In [ ]:
import jsonlines
from tqdm import tqdm
from phase1_distillation import MathDataset, MathRolloutGenerator, AlignmentJudge
from phase1_distillation.dataset import get_problem_id
import phase1_distillation.config as config

os.makedirs(os.path.dirname(config.DRIVE_OUTPUT_FILE), exist_ok=True)

# 1. Khởi tạo Dataset
dataset, processed_ids = MathDataset.load_with_resume(config.DRIVE_PROCESSED_IDS, sample_size=100)

# 2. Khởi tạo Generator & Judge
generator = MathRolloutGenerator(model_id=config.MODEL_ID)
judge = AlignmentJudge(model_id=config.MODEL_ID)

# 3. Vòng lặp chính
with jsonlines.open(config.DRIVE_OUTPUT_FILE, mode='a') as writer, open(config.DRIVE_PROCESSED_IDS, "a") as track_file:
    for item in tqdm(dataset, total=len(dataset), desc="Generating Distillation Data"):
        problem_id = get_problem_id(item['problem'])
        
        # Sinh rollouts (CÓ DÙNG CACHE)
        rollouts = generator.generate(item['problem'], problem_id, cache_dir=config.DRIVE_CACHE_DIR)
        
        # Nếu không đủ 4 câu trả lời, skip để lần sau chạy lại sẽ nạp nốt từ cache
        if len(rollouts) < config.K_ROLLOUTS:
            continue
            
        # Nếu đủ 4 câu, tiến hành tính 6 cặp ma trận
        distance_matrices = judge.evaluate_pairs(item['problem'], rollouts)
        
        if distance_matrices is None or len(distance_matrices) < 6:
            # Nếu tính điểm bị lỗi, vẫn giữ lại rollouts trong cache nhưng chưa đánh dấu processed
            continue
            
        # Ghi kết quả cuối cùng
        result = {
            "question_id": problem_id,
            "question": item['problem'],
            "ground_truth": item['ground_truth_solution'],
            "generated_rollouts": rollouts,
            "distance_matrices": distance_matrices
        }
        writer.write(result)
        
        # Đánh dấu đã xong và XÓA CACHE tạm để sạch ổ đĩa
        track_file.write(problem_id + "\n")
        track_file.flush()
        
        cache_file = os.path.join(config.DRIVE_CACHE_DIR, f"{problem_id}.json")
        if os.path.exists(cache_file):
            os.remove(cache_file)
